# Search Laboratory – A* for Warehouse Robot Navigation

**Contents:** Task 0 (problem formulation) · Task 1 (agent design) · Task 2 (LLM-generated A*) · Task 3 (tests) · Task 4 (code inspection) · Task 5 (BFS vs A*) · Task 6 (heuristics) · Task 7 (evaluation of the LLM) · Final reflection · Appendix (prompts)

## Task 0 – Understand the Search Problem

The problem is the tuple **P = (S, A, T, s₀, G, c)**.

| Component | Specification |
|---|---|
| **State S** | The robot's grid position `(row, col)` on a free cell. |
| **Actions A** | `Up` (−1,0), `Down` (+1,0), `Left` (0,−1), `Right` (0,+1). |
| **Transition T** | `T((r,c), a) = (r+dr, c+dc)` if that cell is inside the grid and not `#`; otherwise the action is not applicable. |
| **Initial state s₀** | The position of `S` in the map, here `(1, 1)`. |
| **Goal G** | The set `{position of 'G'}`, here `{(7, 15)}`. Goal test: `state == goal`. |
| **Cost c** | Every move costs 1, so path cost = number of moves. |

**(a) What information is necessary to specify a state?**
Only the robot's position `(row, col)`. The map is fixed, the environment is static and there is nothing to carry (no battery, no packages), so position alone fully determines what can happen next.

**(b) What makes an action invalid?**
Moving into an obstacle `#`, or outside the grid boundary. (In this map the outer wall is all `#`, but the bounds check is still needed for general maps.)

**(c) Is this a deterministic search problem?**
Yes. Each action in each state has exactly one outcome, the map is fully known and static, and there is a single agent. So the agent can plan offline and execute the plan.

**(d) What would constitute a solution?**
A sequence of valid actions `a₁ … aₙ` that takes the robot from `S` to `G`. An **optimal** solution is one of minimum length (fewest moves, since all costs are 1).

## Task 1 – Plan the Agent (design before prompting)

1. **State representation:** a Python tuple `(row, col)`. Tuples are immutable and hashable, so they can be stored in sets and used as dictionary keys.
2. **Warehouse representation:** a `Grid` class holding a list of lists of characters, plus `start` and `goal` positions found by scanning for `S` and `G`.
3. **Valid actions:** a method `neighbours(state)` tries the four moves and yields `(action, new_state, cost)` only when the new cell is in bounds and not `#`.
4. **Goal recognition:** `is_goal(state)` returns `state == goal`. The test is applied when a state is **removed from the frontier** (this is what guarantees optimality for A*), not when it is first generated.
5. **Frontier contents:** a priority queue (min-heap) of entries `(f, h, counter, g, state)`, where `f = g + h`. `h` and `counter` are tie-breakers, so ties in `f` are never resolved arbitrarily and states are never compared directly.
6. **Path reconstruction:** a `parent` dictionary mapping each state to the state it was reached from. At the goal, follow parents back to `S` and reverse.

**Bookkeeping:** `best_g` (cheapest known cost to each state) and a `closed` set (states already expanded) prevent re-expanding states.

**What the program reports:** whether a solution was found, the path, its length, and the number of states expanded.

## Task 2 – LLM-Generated A* Implementation

### Prompt used (based on the Task 1 design)
> I am implementing a simple goal-based search agent in Python. The environment is a grid represented by an ASCII map. The agent starts at S and must reach G. The symbols # represent obstacles and . represents free cells. The agent can move up, down, left, or right, and every movement has cost 1.
> Implement A* search. Use Manhattan distance as the heuristic: h(n) = |x − xG| + |y − yG|.
> The program should: represent grid positions as states; maintain an appropriate frontier; calculate g(n), h(n) and f(n); avoid repeatedly expanding the same state; reconstruct the path when the goal is reached; report the path and its length; report the number of states expanded.
> Keep the implementation simple and explain the main components of the code.
>
> *(Follow-ups used later: "make the heuristic a parameter", "add a BFS version returning the same result format", "make tie-breaking among equal f-values prefer the state with the smaller h" – see Tasks 5 and 6.)*

### Code

In [10]:
import heapq
import math
from collections import deque

WAREHOUSE = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""

In [11]:
class Grid:
    """The warehouse: a 2D grid plus the search-problem interface
    (initial state, actions/transitions, goal test, step cost)."""

    MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

    def __init__(self, text):
        self.cells = [list(row) for row in text.strip().splitlines()]
        self.rows, self.cols = len(self.cells), len(self.cells[0])
        self.start = self._find("S")
        self.goal = self._find("G")

    def _find(self, ch):
        for r, row in enumerate(self.cells):
            for c, v in enumerate(row):
                if v == ch:
                    return (r, c)
        raise ValueError(f"Map has no '{ch}'")

    def is_free(self, s):
        r, c = s
        return 0 <= r < self.rows and 0 <= c < self.cols and self.cells[r][c] != "#"

    def neighbours(self, s):
        """Yield (action, next_state, step_cost) for every VALID action in state s."""
        for action, (dr, dc) in self.MOVES.items():
            nxt = (s[0] + dr, s[1] + dc)
            if self.is_free(nxt):
                yield action, nxt, 1

    def is_goal(self, s):
        return s == self.goal

    def render(self, path=()):
        canvas = [row[:] for row in self.cells]
        for (r, c) in path:
            if canvas[r][c] == ".":
                canvas[r][c] = "*"
        return "\n".join("".join(row) for row in canvas)


# ---- Heuristics: each takes (state, goal) and estimates the remaining cost ----
def manhattan(s, g):  return abs(s[0] - g[0]) + abs(s[1] - g[1])
def euclidean(s, g):  return math.hypot(s[0] - g[0], s[1] - g[1])
def zero(s, g):       return 0

In [12]:
def astar(grid, h=manhattan, tie_break="h"):
    """
    A* search.  f(n) = g(n) + h(n)
    Returns a dict: found, path, length, expanded.

    tie_break="h"    : among equal f, expand the state with smaller h first
    tie_break="fifo" : among equal f, expand the oldest state first
    """
    start, goal = grid.start, grid.goal
    counter = 0                                    # final tie-breaker; avoids comparing states
    h0 = h(start, goal)
    frontier = [(h0, h0, counter, 0, start)]       # (f, h, counter, g, state) - a min-heap
    parent = {start: None}                         # for path reconstruction
    best_g = {start: 0}                            # cheapest known g for each state
    closed = set()                                 # states already expanded
    expanded = 0

    while frontier:
        f, _, _, g, state = heapq.heappop(frontier)      # lowest f first
        if state in closed:                              # stale duplicate entry -> skip
            continue
        closed.add(state)
        expanded += 1

        if grid.is_goal(state):                          # goal test on EXPANSION
            path, node = [], state
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return {"found": True, "path": path, "length": g, "expanded": expanded}

        for action, nxt, cost in grid.neighbours(state):
            g_new = g + cost
            if nxt not in best_g or g_new < best_g[nxt]: # found a cheaper way to nxt
                best_g[nxt] = g_new
                parent[nxt] = state
                counter += 1
                h_new = h(nxt, goal)
                f_new = g_new + h_new                    # f(n) = g(n) + h(n)
                heapq.heappush(frontier, (f_new, h_new if tie_break == "h" else 0, counter, g_new, nxt))

    return {"found": False, "path": None, "length": None, "expanded": expanded}


def report(grid, result, name="A*"):
    print(f"[{name}] Solution found: {result['found']}")
    print(f"[{name}] States expanded: {result['expanded']}")
    if result["found"]:
        print(f"[{name}] Path length: {result['length']}")
        print(f"[{name}] Path: {result['path']}")
        print(grid.render(result["path"]))

In [13]:
warehouse = Grid(WAREHOUSE)
report(warehouse, astar(warehouse))

[A*] Solution found: True
[A*] States expanded: 64
[A*] Path length: 40
[A*] Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


## Task 3 – Test the Generated Program

### Test 1 – Original warehouse
Run in the previous cell. **Expected:** the map is a winding corridor, so the shortest route is 40 moves (Right×4, Down×4, Right×8, Up×2, Left×6, Up×2, Right×8, Down×6).

The cells below check each test automatically with `assert`, so a failure cannot be missed.

In [14]:
# Test 1: original warehouse
r1 = astar(warehouse)
assert r1["found"] and r1["length"] == 40
assert r1["path"][0] == warehouse.start and r1["path"][-1] == warehouse.goal
assert all(warehouse.is_free(p) for p in r1["path"])                         # no obstacles
assert all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1 for a, b in zip(r1["path"], r1["path"][1:]))  # single steps
print(f"Test 1 passed: length {r1['length']}, expanded {r1['expanded']}")

# Test 2: trivial case - goal adjacent to start
g2 = Grid("#####\n#SG##\n#####")
r2 = astar(g2)
assert r2["found"] and r2["length"] == 1 and r2["path"] == [(1, 1), (1, 2)]
print(f"Test 2 passed: path {r2['path']}, expanded {r2['expanded']}")

# Test 3: unreachable goal - must terminate and report failure
g3 = Grid("#######\n#S....#\n###.###\n#...#G#\n#######")
r3 = astar(g3)
assert r3["found"] is False and r3["path"] is None
print(f"Test 3 passed: no solution reported, expanded {r3['expanded']} states then stopped")

# Test 4: alternative paths - a short route (length 4) and a long route (length 8)
g4 = Grid("#######\n#S...G#\n#.###.#\n#.....#\n#######")
r4 = astar(g4)
assert r4["found"] and r4["length"] == 4
print(f"Test 4 passed: chose the shortest of two routes, length {r4['length']}")
print(g4.render(r4["path"]))

Test 1 passed: length 40, expanded 64
Test 2 passed: path [(1, 1), (1, 2)], expanded 2
Test 3 passed: no solution reported, expanded 9 states then stopped
Test 4 passed: chose the shortest of two routes, length 4
#######
#S***G#
#.###.#
#.....#
#######


### Extra validation – cross-check against an independent algorithm
Hand-made tests only cover cases I thought of. As a stronger test, the cell below generates 500 random grids and compares A*'s answer with **BFS** (which is optimal for unit costs and is a completely different algorithm). Both must agree on *whether* a solution exists and on the *length*.

In [15]:
def bfs(grid):
    """Breadth-first search, returning the same result format as astar()."""
    start = grid.start
    queue, parent, expanded = deque([start]), {start: None}, 0
    while queue:
        state = queue.popleft()
        expanded += 1
        if grid.is_goal(state):
            path, node = [], state
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            return {"found": True, "path": path, "length": len(path) - 1, "expanded": expanded}
        for _, nxt, _ in grid.neighbours(state):
            if nxt not in parent:
                parent[nxt] = state
                queue.append(nxt)
    return {"found": False, "path": None, "length": None, "expanded": expanded}


import random
random.seed(1)
solvable = 0
for _ in range(500):
    R, C = 8, 12
    cells = [["#" if random.random() < 0.3 else "." for _ in range(C)] for _ in range(R)]
    cells[0][0], cells[R-1][C-1] = "S", "G"
    g = Grid("\n".join("".join(row) for row in cells))
    a, b = astar(g), bfs(g)
    assert a["found"] == b["found"], "disagree on solvability"
    if a["found"]:
        solvable += 1
        assert a["length"] == b["length"], "A* not optimal!"
print(f"Cross-check passed on 500 random grids ({solvable} solvable, {500 - solvable} unsolvable).")

Cross-check passed on 500 random grids (248 solvable, 252 unsolvable).


## Task 4 – Inspect the A* Algorithm

| Concept | Where it appears in the code |
|---|---|
| **State** | A `(row, col)` tuple, e.g. `grid.start`, `state`, `nxt`. |
| **Action** | The keys of `Grid.MOVES` (`"Up"`, `"Down"`, `"Left"`, `"Right"`), iterated in `Grid.neighbours`. |
| **Transition** | `Grid.neighbours`: computes `nxt = (s[0]+dr, s[1]+dc)` and yields it only if `is_free(nxt)`. |
| **Goal test** | `grid.is_goal(state)` right after popping a state from the heap. |
| **g(n)** | The `g` value stored in each heap entry; `g_new = g + cost`. Best known values are kept in `best_g`. |
| **h(n)** | The `h` parameter (default `manhattan`), called as `h(nxt, goal)` when a neighbour is generated. |
| **f(n)** | `f_new = g_new + h_new`, the first element of each heap entry. |
| **Frontier** | The list `frontier`, used as a binary min-heap through `heapq.heappush` / `heappop`. |
| **Visited states** | The `closed` set (already expanded), plus `best_g` (a cheaper route can still update an entry). |
| **Path reconstruction** | The `parent` dictionary; on reaching the goal, the loop `while node is not None` walks back to `S` and reverses the list. |

**(a) Data structure for the frontier?** A priority queue implemented as a binary min-heap (`heapq`) of tuples `(f, h, counter, g, state)`.

**(b) How is the next state selected?** `heappop` returns the entry with the smallest `f = g + h`. Ties are broken by smaller `h` (closer to the goal), then by insertion order.

**(c) Where is the heuristic calculated?** In `astar`, at `h_new = h(nxt, goal)` for each generated neighbour (and once for the start). The function itself is `manhattan`.

**(d) Does it explicitly calculate f(n) = g(n) + h(n)?** Yes: `f_new = g_new + h_new`. That value is the primary key of the heap entry.

**(e) How does it prevent repeated exploration?** Three mechanisms. (1) The `closed` set: a popped state already in `closed` is skipped. (2) `best_g`: a neighbour is pushed only if it has never been reached or is reached at strictly lower cost. (3) `parent` is overwritten only in that case. Without these the search could loop forever around cycles and would redo work.

**Design observation.** Because a state can be pushed more than once (whenever a cheaper route is found), the heap can hold stale entries. The `if state in closed: continue` line is what makes this "lazy deletion" safe.

## Task 5 – Compare A* with Blind Search (BFS)
Same warehouse, same start and goal.

In [16]:
def compare(grid, label):
    a, b = astar(grid), bfs(grid)
    print(f"{label}")
    print(f"{'Measure':<18}{'BFS':>8}{'A*':>8}")
    print(f"{'Solution found':<18}{str(b['found']):>8}{str(a['found']):>8}")
    print(f"{'Path length':<18}{b['length']:>8}{a['length']:>8}")
    print(f"{'States expanded':<18}{b['expanded']:>8}{a['expanded']:>8}\n")

compare(warehouse, "Lab warehouse")

Lab warehouse
Measure                BFS      A*
Solution found        True    True
Path length             40      40
States expanded         64      64



**Results on the lab warehouse:**

| Measure | BFS | A* |
|---|---|---|
| Solution found | Yes | Yes |
| Path length | 40 | 40 |
| States expanded | 64 | 64 |

**(a) Did both find a solution?** Yes.
**(b) Same path length?** Yes, both 40, which is optimal. BFS is optimal for unit costs and A* with an admissible heuristic is optimal.
**(c) Which expanded fewer states?** Neither: both expanded all 64 reachable free cells. **A\* did not help on this map.**
**(d) Why might A\* expand fewer states (in general)?** A* uses the heuristic to prefer states that look close to the goal, so it pushes towards `G` instead of exploring evenly in all directions like BFS. It only pays off when the heuristic's guidance matches the geometry.

**Why it did not help here:** the warehouse is a serpentine maze. The optimal route is 40 moves, but the Manhattan distance from `S` to `G` is only 20. The route first has to go *away* from the goal (down, then right, then back up and left...), so the heuristic is badly misleading and A* ends up exploring every corridor whose f ≤ 40, which is the whole map. To show the effect described in the lab's "Think About It" box, the cell below repeats the comparison on an open room with one block in the middle.

In [17]:
OPEN_ROOM = """
###############
#S............#
#.............#
#.....####....#
#.....####....#
#.............#
#............G#
###############
"""
open_room = Grid(OPEN_ROOM)
compare(open_room, "Open room (extra experiment)")

# Why tie-breaking matters: same A*, same heuristic, only the tie rule changes
print("Open room, Manhattan heuristic, expanded states:")
print("  tie-break by smaller h :", astar(open_room, tie_break="h")["expanded"])
print("  tie-break FIFO         :", astar(open_room, tie_break="fifo")["expanded"])

Open room (extra experiment)
Measure                BFS      A*
Solution found        True    True
Path length             17      17
States expanded         70      18

Open room, Manhattan heuristic, expanded states:
  tie-break by smaller h : 18
  tie-break FIFO         : 70


**Open-room result:** A* expanded far fewer states than BFS for the same optimal path (see output above). Here the heuristic really does point towards the goal, so A* mostly follows a nearly straight line, while BFS expands everything within the distance ring.

**Tie-breaking finding.** In an open room with `S` and `G` in opposite corners, *many* cells have the same `f = 17`. With first-in-first-out tie-breaking A* expanded all 70 free cells, i.e. it behaved exactly like BFS. Breaking ties in favour of smaller `h` gave a large reduction. So "A* uses less search" depends on details beyond the heuristic. (This was my first version's behaviour; I found it by running this comparison and fixed it.)

## Task 6 – Investigate the Heuristic

**Why Manhattan distance is appropriate.** With only horizontal/vertical unit-cost moves, each move reduces |Δrow| + |Δcol| by at most 1, so no path can be shorter than the Manhattan distance. Hence `h(n) ≤ h*(n)` everywhere (**admissible**), and it is also **consistent**, so A* with it is optimal. It is also the *tightest* simple bound on an empty grid; on a maze it is still admissible but loose.

**Experiments.** Run each heuristic on both maps. For the h = 2×Manhattan case the lab asks about, I also add 3×, 5× and 10× to see where the behaviour changes.

In [18]:
def scaled(k):
    return lambda s, g: k * manhattan(s, g)

heuristics = [
    ("h = 0",               zero),
    ("Manhattan",           manhattan),
    ("Euclidean",           euclidean),
    ("2 x Manhattan",       scaled(2)),
    ("3 x Manhattan",       scaled(3)),
    ("5 x Manhattan",       scaled(5)),
    ("10 x Manhattan",      scaled(10)),
]

for label, grid in [("Lab warehouse (optimal = 40)", warehouse), ("Open room (optimal = 17)", open_room)]:
    print(label)
    print(f"{'Heuristic':<16}{'Found':>7}{'Length':>8}{'Expanded':>10}")
    for name, h in heuristics:
        r = astar(grid, h)
        print(f"{name:<16}{str(r['found']):>7}{r['length']:>8}{r['expanded']:>10}")
    print()

Lab warehouse (optimal = 40)
Heuristic         Found  Length  Expanded
h = 0              True      40        64
Manhattan          True      40        64
Euclidean          True      40        64
2 x Manhattan      True      40        64
3 x Manhattan      True      48        64
5 x Manhattan      True      48        64
10 x Manhattan     True      48        58

Open room (optimal = 17)
Heuristic         Found  Length  Expanded
h = 0              True      17        70
Manhattan          True      17        18
Euclidean          True      17        54
2 x Manhattan      True      17        18
3 x Manhattan      True      17        18
5 x Manhattan      True      17        18
10 x Manhattan     True      17        18



**Results** (from the cell above; re-run to confirm):

Lab warehouse (optimal length 40):

| Heuristic | Found | Length | Expanded |
|---|---|---|---|
| h = 0 | Yes | 40 | 64 |
| Manhattan | Yes | 40 | 64 |
| Euclidean | Yes | 40 | 64 |
| 2 × Manhattan | Yes | 40 | 64 |
| 3 × Manhattan | Yes | **48** | 64 |
| 5 × Manhattan | Yes | **48** | 64 |
| 10 × Manhattan | Yes | **48** | 58 |

Open room (optimal length 17):

| Heuristic | Found | Length | Expanded |
|---|---|---|---|
| h = 0 | Yes | 17 | 70 |
| Manhattan | Yes | 17 | 18 |
| Euclidean | Yes | 17 | 54 |
| 2 × Manhattan | Yes | 17 | 18 |
| 3–10 × Manhattan | Yes | 17 | 18 |

**Interpretation**
1. **h = 0:** always finds the optimal path (it is admissible), but has no guidance, so A* degenerates to uniform-cost search. It behaved like BFS (70 states in the open room).
2. **Euclidean:** still admissible (a straight line is never longer than a grid path) and still optimal, but it is *smaller* than Manhattan (it underestimates more), so it gives weaker guidance and expanded more states in the open room (54 vs 18). A better (tighter) admissible heuristic prunes more.
3. **Manhattan × 2 (and larger):** no longer admissible since it can overestimate the true remaining cost. On the warehouse, ×2 happened to still return the optimal path, but ×3 and above returned a **48-move path instead of 40**: the search became greedy, committed to the route that looked closest to the goal and accepted the goal on a longer route before the shorter one had been fully considered. On the open room the overestimate did no harm, because there the straight-ish route is also the optimal one. The 2× result on the warehouse is therefore luck and not a guarantee: inadmissibility *permits* suboptimal answers, and the map decides whether it happens.

**Answer to the lab's question:** when the heuristic is too **optimistic** (h = 0, or too small like Euclidean) A* stays optimal but wastes work. When it is too **aggressive** (overestimates) A* can be faster but loses the optimality guarantee. Path lengths that stay at 40 across the first four rows and jump to 48 later are the experimental evidence for this.

## Task 7 – Evaluate the LLM-Generated Agent

1. **What was correct immediately?** The overall structure: the `Grid` class, `heapq` frontier, `closed` set, `parent`-based path reconstruction, and reporting. Test 1–4 passed without changes. The path found was valid and optimal.
2. **Bugs or design problems?** No crash or wrong path, but one *design weakness*: heap entries `(f, counter, ...)` broke ties on `f` by insertion order. That made A* expand as many states as BFS on the open room (70) even though the heuristic was fine. I changed the tie-break to prefer smaller `h`.
3. **How did I discover it?** Not from correctness tests (the path was right); only by comparing the number of expanded states with BFS on a map where A* should have clearly won. Path-only tests would never have caught it.
4. **Unfamiliar terminology?** Terms such as *consistent heuristic*, *lazy deletion* (stale heap entries), *closed set* and *tie-breaking counter* (needed so tuples never compare states). I looked these up / asked the LLM to explain them.
5. **Did I modify the code?** Yes: made the heuristic a parameter, added the tie-break option, added the BFS version with an identical result format, and added the `assert`-based tests.
6. **Most useful tests?** (i) The BFS cross-check on 500 random grids, since it checks optimality and solvability on cases I did not hand-pick; (ii) the unreachable-goal test, since it checks termination; (iii) the BFS-vs-A* state count comparison, which revealed the tie-breaking issue.
7. **Could I have trusted it without testing?** No. It looked right and ran first time, but the tie-breaking weakness only showed up under measurement, and the heuristic experiments show that small changes (a factor 3) silently produce wrong (suboptimal) answers with no error message.
8. **What did I understand about A\* that I didn't before?** That `f = g + h` is a *lower bound on the cost of the best solution through n*; that the goal test belongs at expansion time; that admissibility is what guarantees optimality; that a good heuristic can still be useless on a maze; and that implementation details like tie-breaking change how much work is done.

### Who did what
| | Detail |
|---|---|
| **Designed by me** | Problem formulation (Task 0), agent design (Task 1), the test plan and heuristic experiments. |
| **Suggested by the LLM** | Initial code, use of `heapq`, `closed`/`best_g`/`parent` structure, BFS version, explanations. |
| **Accepted** | Core A* loop, grid/neighbour logic, path reconstruction. |
| **Changed** | Heuristic made a parameter; tie-breaking by `h`; extra reporting; tests added. |
| **Tested** | Original map, trivial map, unreachable goal, alternative paths, 500 random grids vs BFS, BFS vs A* comparison, 7 heuristics on 2 maps. |

## Final Reflection

**1. Why formulate the search problem before writing the search algorithm?**
The formulation (states, actions, transitions, initial state, goal, cost) is what any search algorithm consumes. Once it is precise, the algorithm is generic and can be swapped (BFS, A*, ...) without touching the problem. If the formulation is vague, for example if it is unclear what a state contains, the code encodes hidden assumptions and there is no way to judge whether it is correct. It is also the part an LLM cannot do for you reliably, since you are the one who knows what the real problem is.

**2. In what sense is A\* "informed"?**
Blind searches such as BFS use only the structure of the problem (how many steps from the start). A* additionally uses problem-specific knowledge, a heuristic estimate of the remaining cost to the goal, to decide which state to expand next: it orders the frontier by `f = g + h` instead of by `g` alone. It is informed because of that extra knowledge, and its efficiency depends on how good that knowledge is.

**3. Why does the choice of heuristic matter?**
It affects both correctness and efficiency. An admissible heuristic keeps A* optimal, and the closer it is to the true cost the fewer states are expanded (Manhattan 18 vs Euclidean 54 vs zero 70 in the open room). An inadmissible one can give faster but suboptimal answers (48 instead of 40 on the warehouse). And even a good heuristic can be misleading in a maze, where A* matched BFS (64 states each).

**4. What did the LLM contribute?**
It turned a specification into a working, structured implementation quickly, supplied standard patterns (priority queue, closed set, parent pointers), wrote the BFS variant and explained terminology. It did not decide what the problem was, verify itself, or notice the efficiency problem caused by tie-breaking: those came from design, testing and measurement.

**5. What could go wrong if an engineer accepted LLM code without testing?**
Code that looks plausible and runs can still be wrong or inefficient: returning non-shortest paths, looping forever on unreachable goals, testing the goal at generation instead of expansion (breaking optimality), using an inadmissible heuristic, or wasting work through poor tie-breaking. None of these produce an error message, so *working output ≠ validated algorithm*. Testing with known answers, independent cross-checks and performance measurements is what separates a demonstration from an engineered system.

## Appendix – Prompts used
1. **Initial prompt:** the example prompt from Task 2 above (grid, moves, unit cost, A* with Manhattan heuristic, required outputs).
2. "Make the heuristic a function parameter so I can swap it."
3. "Add a BFS version returning the same result format so I can compare on the same map."
4. "Among states with equal f, prefer the one with the smaller h."
5. "Explain why Manhattan distance is admissible for 4-direction movement on a grid." (Task 6, used only *after* running my own experiments.)